# EP02 — Efficient Frontier & Mean-Variance Optimization
**Quantifaya · Classical Quantitative Finance Series · Episode 2**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Godwin-88/quantifire-web/blob/main/public/notebooks/ep02-efficient-frontier.ipynb)

> **Learning objective:** Derive the Efficient Frontier from first principles, implement constrained quadratic optimization, and understand why mean-variance optimization is fragile in practice.

**Companion post:** [quantifaya.com/blog/ep02-efficient-frontier-free-lunch](https://quantifaya.com/blog/ep02-efficient-frontier-free-lunch)

---
*Quantifaya research notebooks are provided for educational purposes only. Nothing here constitutes financial advice.*

## Learning Objectives

By the end of this notebook you will be able to:

- **Formulate** the mean-variance optimization problem with constraints
- **Solve** the Lagrangian analytically for the unconstrained case
- **Derive** the Minimum Variance Portfolio (MVP) and Tangency Portfolio weights
- **Implement** constrained optimization using `scipy.optimize.minimize`
- **Trace** the full Efficient Frontier by sweeping target returns
- **Monte Carlo simulate** random portfolios to visualize the feasible set
- **Quantify** the error maximization property of MVO
- **Apply** robust alternatives: resampled frontiers, Ledoit-Wolf shrinkage

## Mathematical Prerequisites

### Portfolio Return and Risk

For a portfolio with weights $\mathbf{w}$, expected returns $\boldsymbol{\mu}$, and covariance matrix $\boldsymbol{\Sigma}$:

$$\mu_p = \mathbf{w}^\top \boldsymbol{\mu}$$

$$\sigma_p^2 = \mathbf{w}^\top \boldsymbol{\Sigma} \mathbf{w}$$

### The Optimization Problem

$$\begin{aligned}
\max_{\mathbf{w}} \quad & \mathbf{w}^\top \boldsymbol{\mu} \\
\text{subject to} \quad & \mathbf{w}^\top \boldsymbol{\Sigma} \mathbf{w} \leq \sigma^2_{\text{target}} \\
& \mathbf{1}^\top \mathbf{w} = 1 \\
& w_i \geq 0 \quad \text{(long-only, optional)}
\end{aligned}$$

### Minimum Variance Portfolio (Closed Form)

$$\mathbf{w}_{\text{MVP}} = \frac{\boldsymbol{\Sigma}^{-1} \mathbf{1}}{\mathbf{1}^\top \boldsymbol{\Sigma}^{-1} \mathbf{1}}$$

### Tangency Portfolio (Maximum Sharpe)

$$\mathbf{w}_{\text{tan}} = \frac{\boldsymbol{\Sigma}^{-1} (\boldsymbol{\mu} - r_f \mathbf{1})}{\mathbf{1}^\top \boldsymbol{\Sigma}^{-1} (\boldsymbol{\mu} - r_f \mathbf{1})}$$

## Setup: Imports and Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from scipy.optimize import minimize
from sklearn.covariance import LedoitWolf
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

print("Libraries loaded successfully.")

## 1. Define the Asset Universe

We use a realistic 5-asset universe spanning equities, bonds, gold, real estate, and emerging markets:

In [ ]:
# Asset universe with realistic annual parameters
assets = ['SPY', 'TLT', 'GLD', 'VNQ', 'EEM']
asset_labels = ['S&P 500', 'Long Treasuries', 'Gold', 'Real Estate', 'Emerging Markets']

# Expected annual returns
mu = np.array([0.10, 0.04, 0.06, 0.08, 0.09])

# Annual volatilities
vols = np.array([0.18, 0.08, 0.15, 0.20, 0.22])

# Correlation matrix (realistic estimates)
corr = np.array([
    [1.00, -0.35,  0.05,  0.72,  0.65],
    [-0.35,  1.00,  0.28, -0.20, -0.28],
    [ 0.05,  0.28,  1.00,  0.08,  0.10],
    [ 0.72, -0.20,  0.08,  1.00,  0.55],
    [ 0.65, -0.28,  0.10,  0.55,  1.00],
])

# Build covariance matrix: Σ_ij = σ_i * σ_j * ρ_ij
cov = np.outer(vols, vols) * corr

# Risk-free rate
rf = 0.03

# Display as DataFrame for readability
cov_df = pd.DataFrame(cov, index=assets, columns=assets)
corr_df = pd.DataFrame(corr, index=assets, columns=assets)

print("=== Expected Annual Returns ===")
for asset, ret in zip(assets, mu):
    print(f"  {asset}: {ret:.1%}")

print("\n=== Annual Volatilities ===")
for asset, vol in zip(assets, vols):
    print(f"  {asset}: {vol:.1%}")

print("\n=== Correlation Matrix ===")
print(corr_df.round(2))

## 2. Portfolio Statistics Functions

Core functions for computing portfolio return, volatility, and Sharpe ratio:

In [ ]:
def portfolio_return(weights, mu):
    """Expected portfolio return."""
    return np.dot(weights, mu)

def portfolio_volatility(weights, cov):
    """Portfolio volatility (standard deviation)."""
    return np.sqrt(np.dot(weights.T, np.dot(cov, weights)))

def portfolio_sharpe(weights, mu, cov, rf=0.03):
    """Sharpe Ratio of the portfolio."""
    ret = portfolio_return(weights, mu)
    vol = portfolio_volatility(weights, cov)
    return (ret - rf) / vol if vol > 0 else 0

def print_portfolio_weights(weights, assets, mu, cov, rf=0.03, label=""):
    """Pretty-print portfolio weights and metrics."""
    if label:
        print(f"\n{'='*50}")
        print(f"{label}")
        print(f"{'='*50}")
    
    print("\nWeights:")
    for asset, w in zip(assets, weights):
        if w > 0.001:  # Only show non-zero weights
            print(f"  {asset}: {w:>7.1%}")
    
    ret = portfolio_return(weights, mu)
    vol = portfolio_volatility(weights, cov)
    sharpe = portfolio_sharpe(weights, mu, cov, rf)
    
    print(f"\nExpected Return:  {ret:.2%}")
    print(f"Volatility:       {vol:.2%}")
    print(f"Sharpe Ratio:     {sharpe:.3f}")

## 3. Analytical Solutions

### 3.1 Minimum Variance Portfolio (Closed Form)

The MVP requires no return estimates — it's purely a function of the covariance structure:

In [ ]:
def mvp_analytical(cov):
    """Compute MVP weights analytically."""
    cov_inv = np.linalg.inv(cov)
    ones = np.ones(len(cov))
    w = cov_inv @ ones / (ones @ cov_inv @ ones)
    return w

def tangency_analytical(mu, cov, rf):
    """Compute tangency (max Sharpe) portfolio weights analytically."""
    cov_inv = np.linalg.inv(cov)
    ones = np.ones(len(mu))
    excess_mu = mu - rf
    w = cov_inv @ excess_mu / (ones @ cov_inv @ excess_mu)
    return w

# Compute analytical solutions
w_mvp = mvp_analytical(cov)
w_tangency = tangency_analytical(mu, cov, rf)

print_portfolio_weights(w_mvp, assets, mu, cov, rf, "Minimum Variance Portfolio (Analytical)")
print_portfolio_weights(w_tangency, assets, mu, cov, rf, "Tangency Portfolio (Analytical)")

### 3.2 Constrained Optimization with scipy

In practice we need constraints (weights sum to 1, long-only). We use SLSQP:

In [ ]:
# Optimization constraints and bounds
constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0}]  # Weights sum to 1
bounds = tuple((0.0, 1.0) for _ in range(len(assets)))  # Long-only
init_weights = np.ones(len(assets)) / len(assets)  # Equal weight start

# Minimize Variance Portfolio
def neg_portfolio_return(weights, mu):
    return -portfolio_return(weights, mu)

def minimize_variance(weights, cov):
    return portfolio_volatility(weights, cov)

result_mvp = minimize(
    minimize_variance,
    init_weights,
    args=(cov,),
    method='SLSQP',
    bounds=bounds,
    constraints=constraints
)

w_mvp_constrained = result_mvp.x
print_portfolio_weights(w_mvp_constrained, assets, mu, cov, rf, "Min Variance Portfolio (Constrained)")

# Maximize Sharpe (Tangency) Portfolio
def neg_sharpe(weights, mu, cov, rf=0.03):
    return -portfolio_sharpe(weights, mu, cov, rf)

result_sharpe = minimize(
    neg_sharpe,
    init_weights,
    args=(mu, cov, rf),
    method='SLSQP',
    bounds=bounds,
    constraints=constraints
)

w_sharpe = result_sharpe.x
print_portfolio_weights(w_sharpe, assets, mu, cov, rf, "Max Sharpe Portfolio (Constrained)")

## 4. Monte Carlo: Visualizing the Feasible Set

Generate 10,000 random portfolios to see the full feasible region:

In [ ]:
np.random.seed(42)
n_portfolios = 10000

# Generate random weights (Dirichlet distribution for valid weight vectors)
random_weights = np.random.dirichlet(np.ones(len(assets)), n_portfolios)

# Compute metrics for all random portfolios
mc_returns = np.array([portfolio_return(w, mu) for w in random_weights])
mc_vols = np.array([portfolio_volatility(w, cov) for w in random_weights])
mc_sharpes = np.array([portfolio_sharpe(w, mu, cov, rf) for w in random_weights])

# Plot feasible set
fig, ax = plt.subplots(figsize=(12, 8))

# Scatter plot of random portfolios, colored by Sharpe
scatter = ax.scatter(mc_vols, mc_returns, c=mc_sharpes, cmap='viridis', 
                     s=20, alpha=0.6, edgecolors='none', label='Random Portfolios')

# MVP point
mvp_vol = portfolio_volatility(w_mvp_constrained, cov)
mvp_ret = portfolio_return(w_mvp_constrained, mu)
ax.scatter(mvp_vol, mvp_ret, c='red', s=200, marker='*', 
           edgecolors='white', linewidths=2, zorder=5, label='Min Variance')

# Max Sharpe point
sharpe_vol = portfolio_volatility(w_sharpe, cov)
sharpe_ret = portfolio_return(w_sharpe, mu)
ax.scatter(sharpe_vol, sharpe_ret, c='orange', s=200, marker='*',
           edgecolors='white', linewidths=2, zorder=5, label='Max Sharpe')

ax.set_xlabel('Annual Volatility', fontsize=14)
ax.set_ylabel('Annual Expected Return', fontsize=14)
ax.set_title('Feasible Portfolio Set (10,000 Random Portfolios)', fontsize=16)
ax.legend(loc='lower right', fontsize=12)
ax.grid(True, alpha=0.3)

# Add colorbar
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Sharpe Ratio', fontsize=12)

plt.tight_layout()
plt.show()

print(f"Best random portfolio Sharpe: {mc_sharpes.max():.3f}")
print(f"Max Sharpe portfolio Sharpe:  {portfolio_sharpe(w_sharpe, mu, cov, rf):.3f}")

## 5. Tracing the Efficient Frontier

For each target return, find the minimum variance portfolio. This traces out the frontier:

In [ ]:
def optimize_for_target_return(target_return, mu, cov, bounds, constraints_base):
    """Find minimum variance portfolio for a given target return."""
    constraints = constraints_base + [
        {'type': 'eq', 'fun': lambda w, mu=mu, target=target_return: portfolio_return(w, mu) - target}
    ]
    result = minimize(
        minimize_variance,
        init_weights,
        args=(cov,),
        method='SLSQP',
        bounds=bounds,
        constraints=constraints
    )
    return result.x if result.success else None

# Sweep target returns from MVP return to max individual asset return
mvp_ret = portfolio_return(w_mvp_constrained, mu)
max_ret = mu.max()
target_returns = np.linspace(mvp_ret, max_ret, 50)

frontier_weights = []
frontier_vols = []
frontier_returns = []
frontier_sharpes = []

for target in target_returns:
    w = optimize_for_target_return(target, mu, cov, bounds, constraints)
    if w is not None:
        vol = portfolio_volatility(w, cov)
        ret = portfolio_return(w, mu)
        sharpe = portfolio_sharpe(w, mu, cov, rf)
        frontier_weights.append(w)
        frontier_vols.append(vol)
        frontier_returns.append(ret)
        frontier_sharpes.append(sharpe)

frontier_weights = np.array(frontier_weights)
frontier_vols = np.array(frontier_vols)
frontier_returns = np.array(frontier_returns)
frontier_sharpes = np.array(frontier_sharpes)

# Plot the Efficient Frontier
fig, ax = plt.subplots(figsize=(12, 8))

# Random portfolios (background)
ax.scatter(mc_vols, mc_returns, c='gray', s=15, alpha=0.3, label='Inefficient Portfolios')

# Efficient Frontier
ax.plot(frontier_vols, frontier_returns, 'b-', linewidth=3, label='Efficient Frontier')
ax.fill_betweenx(frontier_returns, frontier_vols, mc_vols.min(), 
                 color='blue', alpha=0.1, label='Efficient Region')

# Special points
ax.scatter(mvp_vol, mvp_ret, c='red', s=200, marker='*', 
           edgecolors='white', linewidths=2, zorder=5, label='Min Variance')
ax.scatter(sharpe_vol, sharpe_ret, c='orange', s=200, marker='*',
           edgecolors='white', linewidths=2, zorder=5, label='Max Sharpe')

# Capital Market Line (from risk-free rate through tangency)
cml_vols = np.linspace(0, sharpe_vol * 1.3, 100)
cml_returns = rf + (sharpe_ret - rf) / sharpe_vol * cml_vols
ax.plot(cml_vols, cml_returns, 'g--', linewidth=2, alpha=0.7, label='Capital Market Line')

ax.set_xlabel('Annual Volatility', fontsize=14)
ax.set_ylabel('Annual Expected Return', fontsize=14)
ax.set_title('Efficient Frontier', fontsize=16)
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Efficient Frontier: {len(frontier_vols)} portfolios computed")
print(f"Frontier return range: {frontier_returns.min():.2%} to {frontier_returns.max():.2%}")
print(f"Frontier volatility range: {frontier_vols.min():.2%} to {frontier_vols.max():.2%}")

## 6. Portfolio Weight Evolution Along the Frontier

See how optimal weights shift as we move along the frontier:

In [ ]:
# Stack frontier weights for visualization
fig, ax = plt.subplots(figsize=(14, 8))

# Create stacked area chart
x = np.arange(len(frontier_weights))
bottom = np.zeros(len(frontier_weights))

colors = plt.cm.Set2(np.linspace(0, 1, len(assets)))
for i, (asset, color) in enumerate(zip(assets, colors)):
    ax.fill_between(x, bottom, bottom + frontier_weights[:, i], 
                    color=color, alpha=0.8, label=f'{asset}')
    bottom += frontier_weights[:, i]

ax.set_xlabel('Portfolio Index (Low → High Return)', fontsize=14)
ax.set_ylabel('Portfolio Weight', fontsize=14)
ax.set_title('Asset Allocation Along the Efficient Frontier', fontsize=16)
ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=11)
ax.set_xticks([0, len(frontier_weights) // 2, len(frontier_weights) - 1])
ax.set_xticklabels(['Low Risk', 'Medium Risk', 'High Return'])
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print weight allocation at key points
print("\n=== Weight Allocation at Key Frontier Points ===\n")
print(f"{'Asset':<8} {'Min Var':<10} {'Mid':<10} {'Max Return':<12}")
print("-" * 40)
for i, asset in enumerate(assets):
    min_var_w = frontier_weights[0, i]
    mid_w = frontier_weights[len(frontier_weights) // 2, i]
    max_ret_w = frontier_weights[-1, i]
    print(f"{asset:<8} {min_var_w:>8.1%}   {mid_w:>8.1%}   {max_ret_w:>8.1%}")

## 7. The Error Maximization Problem

Demonstrate how sensitive MVO is to return forecasts:

In [ ]:
print("=== Sensitivity Analysis: Error Maximization ===\n")

# Perturb expected returns by small amounts
perturbations = [0.0, 0.005, 0.01, 0.02, 0.05]

for p in perturbations:
    mu_perturbed = mu * (1 + p)
    
    result = minimize(
        neg_sharpe,
        init_weights,
        args=(mu_perturbed, cov, rf),
        method='SLSQP',
        bounds=bounds,
        constraints=constraints
    )
    w_perturbed = result.x
    
    # Compute weight changes from baseline
    weight_change = np.abs(w_perturbed - w_sharpe).max()
    
    print(f"Return forecast perturbation: +{p:.1%}")
    print(f"  Max weight change: {weight_change:.1%}")
    print(f"  New Sharpe: {portfolio_sharpe(w_perturbed, mu, cov, rf):.3f}")
    print()

**Key insight:** Even a 1% perturbation in return forecasts (well within estimation error) can cause weight swings of 5-15%. This is why pure MVO is rarely used in production without robustness enhancements.

## 8. Robust Alternatives

### 8.1 Ledoit-Wolf Shrinkage Estimator

Replace the sample covariance with a shrinkage estimator that blends with a structured target:

In [ ]:
# Simulate historical returns to demonstrate shrinkage
np.random.seed(42)
n_observations = 252 * 3  # 3 years of daily data

# Generate correlated returns using Cholesky decomposition
L = np.linalg.cholesky(cov / 252)  # Convert annual cov to daily
daily_mu = mu / 252
z = np.random.randn(n_observations, len(assets))
daily_returns = daily_mu + z @ L.T

# Sample covariance (naive)
cov_sample = np.cov(daily_returns.T) * 252

# Ledoit-Wolf shrinkage
lw = LedoitWolf()
lw.fit(daily_returns)
cov_lw = lw.covariance_ * 252
shrinkage_intensity = lw.shrinkage_

print("=== Covariance Estimation Comparison ===\n")
print(f"Shrinkage intensity (α): {shrinkage_intensity:.3f}")
print(f"  0 = use sample, 1 = use structured target")

# Compare portfolio metrics with different covariance estimators
def optimize_with_cov(cov_est, label):
    result = minimize(
        lambda w: portfolio_volatility(w, cov_est),
        init_weights,
        args=(),
        method='SLSQP',
        bounds=bounds,
        constraints=constraints
    )
    w = result.x
    vol = portfolio_volatility(w, cov_est)
    ret = portfolio_return(w, mu)
    print(f"\n{label}:")
    print(f"  Volatility: {vol:.2%}")
    print(f"  Return: {ret:.2%}")
    return w

print("\n--- MVP with Different Covariance Estimators ---")
w_sample = optimize_with_cov(cov_sample, "Sample Covariance")
w_lw = optimize_with_cov(cov_lw, "Ledoit-Wolf Shrinkage")
w_true = optimize_with_cov(cov, "True Covariance (oracle)")

# Compare weight differences
print("\n=== Weight Differences vs True MVP ===")
print(f"{'Asset':<8} {'Sample Δ':<12} {'LW Δ':<12}")
print("-" * 32)
for i, asset in enumerate(assets):
    sample_diff = abs(w_sample[i] - w_true[i])
    lw_diff = abs(w_lw[i] - w_true[i])
    print(f"{asset:<8} {sample_diff:>8.1%}      {lw_diff:>8.1%}")

### 8.2 Resampled Efficient Frontier (Michaud)

Bootstrap multiple samples, compute frontiers, and average:

In [ ]:
def resampled_frontier(daily_returns, mu_annual, cov_annual, rf, n_bootstrap=500):
    """Compute resampled efficient frontier."""
    n_assets = daily_returns.shape[1]
    n_obs = daily_returns.shape[0]
    
    all_frontier_vols = []
    all_frontier_returns = []
    
    for boot in range(n_bootstrap):
        # Bootstrap sample
        indices = np.random.choice(n_obs, size=n_obs, replace=True)
        boot_returns = daily_returns[indices]
        
        # Estimate parameters from bootstrap sample
        mu_boot = boot_returns.mean(axis=0) * 252
        cov_boot = np.cov(boot_returns.T) * 252
        
        # Compute MVP for this sample
        try:
            w_boot = mvp_analytical(cov_boot)
            w_boot = np.clip(w_boot, 0, 1)  # Enforce long-only
            w_boot = w_boot / w_boot.sum()  # Re-normalize
            
            vol = portfolio_volatility(w_boot, cov_annual)
            ret = portfolio_return(w_boot, mu_annual)
            all_frontier_vols.append(vol)
            all_frontier_returns.append(ret)
        except:
            continue
    
    return np.array(all_frontier_vols), np.array(all_frontier_returns)

# Compute resampled frontier
print("Computing resampled frontier (500 bootstraps)...")
resample_vols, resample_returns = resampled_frontier(daily_returns, mu, cov, rf, n_bootstrap=500)

# Plot comparison
fig, ax = plt.subplots(figsize=(12, 8))

# Resampled frontier (scatter)
ax.scatter(resample_vols, resample_returns, c='purple', s=30, 
           alpha=0.5, label='Resampled MVP (500 bootstraps)')

# Original frontier
ax.plot(frontier_vols, frontier_returns, 'b-', linewidth=3, 
        label='Original Efficient Frontier')

# Original MVP
ax.scatter(mvp_vol, mvp_ret, c='red', s=200, marker='*',
           edgecolors='white', linewidths=2, zorder=5, label='Original MVP')

# Average resampled MVP
avg_vol = resample_vols.mean()
avg_ret = resample_returns.mean()
ax.scatter(avg_vol, avg_ret, c='yellow', s=300, marker='D',
           edgecolors='black', linewidths=2, zorder=6, label='Average Resampled MVP')

ax.set_xlabel('Annual Volatility', fontsize=14)
ax.set_ylabel('Annual Expected Return', fontsize=14)
ax.set_title('Resampled Efficient Frontier vs Original', fontsize=16)
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n=== Resampled Frontier Statistics ===")
print(f"Number of valid bootstrap portfolios: {len(resample_vols)}")
print(f"Average MVP volatility: {avg_vol:.2%}")
print(f"Average MVP return: {avg_ret:.2%}")
print(f"Original MVP volatility: {mvp_vol:.2%}")
print(f"Original MVP return: {mvp_ret:.2%}")

## 9. Summary Dashboard

Consolidate all key portfolios into a comparison table:

In [ ]:
# Portfolio comparison
portfolios = {
    'Equal Weight': init_weights,
    'Min Variance': w_mvp_constrained,
    'Max Sharpe': w_sharpe,
    'MVP (LW Shrinkage)': w_lw,
}

comparison = []
for name, w in portfolios.items():
    ret = portfolio_return(w, mu)
    vol = portfolio_volatility(w, cov)
    sharpe = portfolio_sharpe(w, mu, cov, rf)
    comparison.append({
        'Portfolio': name,
        'Return': ret,
        'Volatility': vol,
        'Sharpe': sharpe,
        **{f'w_{assets[i]}': w[i] for i in range(len(assets))}
    })

comparison_df = pd.DataFrame(comparison)
comparison_df = comparison_df.set_index('Portfolio')

# Format for display
display_df = comparison_df.copy()
for col in ['Return', 'Volatility', 'Sharpe']:
    if col == 'Sharpe':
        display_df[col] = display_df[col].map('{:.3f}'.format)
    else:
        display_df[col] = display_df[col].map('{:.2%}'.format)

weight_cols = [f'w_{a}' for a in assets]
for col in weight_cols:
    display_df[col] = display_df[col].map('{:.1%}'.format)

print("\n=== Portfolio Comparison ===")
print(display_df.to_string())

## Key Takeaways

1. **The Efficient Frontier is the set of optimal portfolios** — for each risk level, it gives the maximum achievable return.

2. **Two special portfolios anchor the frontier:**
   - **Minimum Variance Portfolio:** Leftmost point, requires no return estimates
   - **Tangency Portfolio:** Highest Sharpe ratio, most sensitive to return forecasts

3. **MVO is fragile:** Small errors in return estimates cause large weight swings (error maximization).

4. **Robust alternatives help:**
   - Ledoit-Wolf shrinkage produces better covariance estimates
   - Resampled frontiers smooth out estimation noise
   - MVP is often more stable than max Sharpe out-of-sample

5. **Practical recommendation:** Use the framework to ask "am I on the frontier?" but prefer robust methods for actual weight computation.

---

**References:**

- Markowitz, H. (1952). "Portfolio Selection." *Journal of Finance*, 7(1), 77–91.
- Michaud, R.O. (1989). "The Markowitz Optimization Enigma." *Financial Analysts Journal*, 45(1), 31–42.
- Ledoit, O. & Wolf, M. (2004). "Honey, I Shrunk the Sample Covariance Matrix." *JPM*, 30(4), 110–119.

---
*Quantifaya — Quantitative Finance for Web2 & Web3. Not financial advice.*